**silver layer requirement**
- Remove duplicate orders 
- Remove invalid customers 
- Replace null quantity 
- Remove negative prices 
- Convert order date 
- Calculate Total Amount 

In [0]:
df = spark.table("associate_assignment.default.bronze_orders")

# cast column data type to proper formate

df = (
    df
    .withColumn("order_id", col("order_id").cast("string"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    # .withColumn("product_id", col("product_id").cast("string"))
    .withColumn("order_date", col("order_date").cast("string"))
    .withColumn("quantity", col("quantity").cast("double").cast("int"))
    .withColumn("price", col("price").cast("decimal(10,2)"))
)

df.printSchema()

df.groupBy("order_id").count().filter("count > 1").show() # before removing duplicates

df_distinct = df.dropDuplicates(["order_id"])

df_distinct.groupBy("order_id").count().filter("count > 1").show() # aftr removing duplicates



### Remove invalid customers

In [0]:
# before removing invalid customers
df_customer_invalid = df_distinct.filter(col("customer_id").isNull() | col("customer_id").isin("0","UNKNOWN"))

display(df_customer_invalid) 

#after removing invalid customers

df_customer_valid = df_distinct.filter(
    col("customer_id").isNotNull() &
    (col("customer_id") != "0") & (col("customer_id") != "UNKNOWN")
)

display(df_customer_valid)


### Replace null quantity

In [0]:
# before replace null quantity
df_null_quantity = df_customer_valid.filter(col("quantity").isNull())

display(df_null_quantity)

In [0]:
df_lkp_product = spark.table("associate_assignment.look_up_tables.product")

df_order_product = df_customer_valid.join(df_lkp_product, col("product") == col("product_name"), "inner")

df_order_product.createOrReplaceTempView("vw_order_product")

df_valid_quantity = spark.sql("""SELECT
    order_id,
    customer_id,
    product_id,
    CASE
        WHEN quantity IS NULL THEN round(price / unit_price,0)
        ELSE quantity
    END AS quantity,
    price,
    order_date,
    _rescued_data,
    ingestion_timestamp,
    source_file
FROM vw_order_product""")

### Remove negative prices

In [0]:
#before removing negative prices
df_negative_price = df_valid_quantity.filter(
    col("price") < 0
)

# display(df_negative_price)


#after removing negative prices
df_valid_price =  df_valid_quantity.filter(
    col("price") > 0
)

print("before removing negative price value",df_valid_quantity.count())
print("negative price value : " ,df_negative_price.count())
print("valid price value : " ,df_valid_price.count())




### Convert order date

In [0]:
display(df_valid_price)

from pyspark.sql.functions import to_date, col,coalesce,expr

df_valid_date_formate = df_valid_price.withColumn(
    "order_date",
    coalesce(
        expr("CAST(try_to_timestamp(order_date, 'yyyy-MM-dd') AS DATE)"),
        expr("CAST(try_to_timestamp(order_date, 'dd/MM/yyyy') AS DATE)"),
        expr("CAST(try_to_timestamp(order_date, 'MM-dd-yyyy') AS DATE)")
    )
)

display(df_valid_date_formate)

In [0]:
display(df_valid_date_formate.agg({"price": "sum"}))

In [0]:
df_valid_date_formate.write.mode("overwrite").saveAsTable("associate_assignment.default.silver_orders")